# Peeking: vì sao nhìn kết quả mỗi ngày làm hỏng thí nghiệm

Bạn sẽ mô phỏng các thí nghiệm **A/A** (hai nhóm giống hệt nhau, không có tác động nào), nhìn
kết quả nhiều lần trong lúc chạy, và đo tỉ lệ báo "có ý nghĩa" nhầm. Sau đó tự hiệu chỉnh biên
O'Brien-Fleming để nhìn nhiều lần mà vẫn giữ false positive ở 5%.

**Mô hình mô phỏng** (dùng chung cho cả bài):
- Mỗi thí nghiệm có `n_per_group` quan sát mỗi nhóm, giá trị `Normal(0, 1)`, rút bằng
  `rng = np.random.default_rng(seed)`.
- Nhìn ở `n_looks` mốc cách đều: lần nhìn thứ k (k = 1..n_looks) dùng `n_k = n_per_group * k // n_looks`
  quan sát đầu tiên mỗi nhóm.
- Thống kê z với variance đã biết bằng 1: `z_k = (mean_B − mean_A) / sqrt(2 / n_k)`.

Chấm: `uv run pytest tracks/data-science/modules/03-ab-testing/exercises/intermediate/03-peeking -q`

In [ ]:
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm

## 1. Hàm phụ: z ở mỗi lần nhìn

Gợi ý vector hoá: rút cả ma trận `(n_sims, n_per_group)` cho mỗi nhóm một lần, dùng `cumsum`
theo trục quan sát để có tổng tích luỹ, rồi lấy cột tại các mốc `n_k`.

In [ ]:
def z_at_looks(n_per_group: int, n_looks: int, n_sims: int, seed: int = 0) -> np.ndarray:
    """Ma trận (n_sims, n_looks): z của từng thí nghiệm A/A ở từng lần nhìn."""
    rng = np.random.default_rng(seed)
    a = rng.standard_normal((n_sims, n_per_group)).cumsum(axis=1)
    b = rng.standard_normal((n_sims, n_per_group)).cumsum(axis=1)
    n_k = np.array([n_per_group * k // n_looks for k in range(1, n_looks + 1)])
    diff = (b[:, n_k - 1] - a[:, n_k - 1]) / n_k
    return diff / np.sqrt(2 / n_k)

## 2. Tỉ lệ false positive khi peeking

Bác bỏ H0 nếu **bất kỳ** lần nhìn nào có p-value hai phía < alpha.

In [ ]:
def peeking_fpr(n_per_group: int, n_looks: int, n_sims: int, alpha: float = 0.05, seed: int = 0) -> float:
    z = z_at_looks(n_per_group, n_looks, n_sims, seed)
    return float(np.mean((np.abs(z) > norm.ppf(1 - alpha / 2)).any(axis=1)))

In [ ]:
# explore: false positive tăng theo số lần nhìn
for k in (1, 2, 5, 10, 20):
    print(k, peeking_fpr(n_per_group=500, n_looks=k, n_sims=5000))

In [ ]:
# explore: đường p-value của vài thí nghiệm A/A theo thời gian
import matplotlib.pyplot as plt

z = z_at_looks(n_per_group=2000, n_looks=50, n_sims=5, seed=11)
plt.plot(np.arange(1, 51) / 50, 2 * norm.sf(np.abs(z.T)))
plt.axhline(0.05, color="red", linestyle="--")
plt.xlabel("tỉ lệ dữ liệu đã thu")
plt.ylabel("p-value")
plt.yscale("log")
plt.show()

## 3. Biên O'Brien-Fleming

Lần nhìn thứ k bác bỏ khi `|z_k| > c * sqrt(n_looks / k)`: lần đầu cần bằng chứng áp đảo, lần cuối
biên bằng `c`. Hằng số `c` được chọn để tổng false positive qua mọi lần nhìn bằng alpha.

In [ ]:
def obf_fpr(c: float, n_per_group: int, n_looks: int, n_sims: int, seed: int = 0) -> float:
    """Tỉ lệ thí nghiệm A/A bị bác bỏ ở ít nhất một lần nhìn với biên O'Brien-Fleming hằng số c."""
    z = z_at_looks(n_per_group, n_looks, n_sims, seed)
    bounds = c * np.sqrt(n_looks / np.arange(1, n_looks + 1))
    return float(np.mean((np.abs(z) > bounds).any(axis=1)))

In [ ]:
def calibrate_obf_constant(n_looks: int, alpha: float = 0.05, n_sims: int = 20000, seed: int = 0) -> float:
    """Tìm c sao cho obf_fpr(c, n_per_group=100, n_looks, n_sims, seed) ≈ alpha."""
    z = np.abs(z_at_looks(100, n_looks, n_sims, seed))
    # max_k |z_k| / sqrt(K/k) vượt c  ⇔  thí nghiệm bị bác bỏ, nên c là phân vị 1 − alpha của nó
    scaled = (z / np.sqrt(n_looks / np.arange(1, n_looks + 1))).max(axis=1)
    return float(brentq(lambda c: np.mean(scaled > c) - alpha, 1.5, 4.0))

In [ ]:
# explore: so với 1.96 cố định ở mọi lần nhìn
c = calibrate_obf_constant(5)
print(c, [round(c * np.sqrt(5 / k), 2) for k in range(1, 6)])

## 4. Trả lời

1. Cho PM dashboard hằng ngày nhưng chỉ gồm **SRM và guardrail an toàn** (crash, lỗi thanh toán),
   với quy tắc dừng vì an toàn viết trước. Metric chính chỉ đọc ở các mốc định trước (group
   sequential) hoặc dùng phương pháp always-valid (mSPRT / confidence sequence) nếu platform hỗ trợ.
2. Biên cuối là `c ≈ 2.04` thay vì 1.96, nên với cùng cỡ mẫu, power ở lần nhìn cuối thấp hơn một
   chút (cần thêm khoảng 2–3% mẫu để bù). Đổi lại được phép dừng sớm khi tác động rất lớn, tiết
   kiệm thời gian trung bình đáng kể.